# 02 — Well State Detection

**Goals**
- Q2: characterise acceleration when running vs stopped; detect state
- Q3: classify controller mode (continuous / timer / shutdown)
- Q4: estimate runtime fraction (excluding >12 h shutdowns)

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from well_analysis.data import load_test_data
from well_analysis.signal import check_even_sampling
from well_analysis.detection import detect_well_state, classify_controller_mode
from well_analysis.viz import plot_time_series, plot_well_state

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

df = load_test_data()
_, fs = check_even_sampling(df['Timestamp'])
print(f"fs = {fs:.2f} Hz")

## 1. What to expect: running vs stopped (Q2)

- **Stopped**: rod is stationary → acceleration ≈ constant (gravity + sensor offset, ≈ −9.81 m/s²).
- **Running**: rod oscillates at the pump frequency (0.05–0.1 Hz) → acceleration varies periodically.

→ **Method**: compute the rolling RMS of the AC component (signal minus its median). High RMS = running; near-zero RMS = stopped. O(n) via `scipy.ndimage.uniform_filter1d`.

In [ ]:
accel = df['Acceleration'].values
is_running = detect_well_state(accel, fs=fs)

print(f"Fraction running (raw): {is_running.mean():.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 5), sharex=True)
plot_time_series(df['Timestamp'], accel, ylabel='m/s²',
                 title='Acceleration', ax=axes[0])
plot_well_state(df['Timestamp'], is_running, ax=axes[1])
plt.tight_layout()
plt.show()

## 2. Controller mode classification (Q3)

In [ ]:
segments = classify_controller_mode(df['Timestamp'], is_running)
print(segments['mode'].value_counts())
segments.head(20)

## 3. Runtime fraction — excluding long shutdowns (Q4)

In [ ]:
# Exclude segments where the well was down > 12 h
shutdown_mask = (segments['mode'] == 'shutdown')
excluded_time_h = segments.loc[shutdown_mask, 'duration_h'].sum()

total_h = (df['Timestamp'].iloc[-1] - df['Timestamp'].iloc[0]).total_seconds() / 3600
active_h = total_h - excluded_time_h

running_h = segments.loc[segments['running'] == True, 'duration_h'].sum()
runtime_fraction = running_h / active_h

print(f"Total span          : {total_h:.1f} h")
print(f"Excluded (shutdown) : {excluded_time_h:.1f} h")
print(f"Active window       : {active_h:.1f} h")
print(f"Running time        : {running_h:.1f} h")
print(f"Runtime fraction    : {runtime_fraction:.3f}  ({runtime_fraction*100:.1f} %)")